# 5. Remotes and origin

A **remote** is another copy of your repository, usually on a server like
GitHub. **origin** is the usual name for it.

Here we build our own "server" in a folder, so you can see there is no
magic. Three folders:

```mermaid
flowchart LR
    me["barge (you)"] -->|git push| origin["barge.git (the server)"]
    origin -->|git fetch| me
    origin -->|git clone| deniz["deniz (a colleague)"]
    deniz -->|git push| origin
```

In [1]:
import os, shutil, stat, sys
from pathlib import Path


def remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)  # git makes its object files read-only
    func(path)


work = Path.home() / "git-course" / "05-remotes"
if work.exists():
    shutil.rmtree(work, onexc=remove_readonly)
work.mkdir(parents=True)
%cd {work}

/home/emircem/git-course/05-remotes


## The server

A server repository is **bare**: it has the history but no working files,
because nobody edits files on a server.

In [2]:
!git init --bare barge.git

Initialized empty Git repository in /home/emircem/git-course/05-remotes/barge.git/


## Your repository

In [3]:
os.mkdir("barge")
%cd barge

/home/emircem/git-course/05-remotes/barge


In [4]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)

Writing hydrostatics.py


In [5]:
!git init
!git add hydrostatics.py
!git commit -m "Add volume and displacement for a box barge"

Initialized empty Git repository in /home/emircem/git-course/05-remotes/barge/.git/
[main (root-commit) 2dbd296] Add volume and displacement for a box barge
 1 file changed, 13 insertions(+)
 create mode 100644 hydrostatics.py


## Adding a remote

`git remote add` gives the server a name. The name is `origin` by
convention. On GitHub the address would look like
`https://github.com/you/barge.git`; here it is just a folder.

In [6]:
!git remote add origin ../barge.git

In [7]:
!git remote -v

origin	../barge.git (fetch)
origin	../barge.git (push)


## Pushing

`git push` sends your commits to the server. `-u` is needed the first time
only: it links your `main` to the server's `main`.

In [8]:
!git push -u origin main

Enumerating objects: 3, done.
Counting objects: 100% (3/3), done.
Delta compression using up to 20 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 441 bytes | 441.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
To ../barge.git
 * [new branch]      main -> main
branch 'main' set up to track 'origin/main'.


In [9]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


## A colleague clones it

`git clone` copies a repository from the server, including all its history.
The clone knows `origin` automatically.

In [10]:
%cd ..
!git clone barge.git deniz

/home/emircem/git-course/05-remotes
Cloning into 'deniz'...
done.


In [11]:
%cd deniz
!git config user.name "Deniz"
!git log --oneline

/home/emircem/git-course/05-remotes/deniz
2dbd296 (HEAD -> main, origin/main, origin/HEAD) Add volume and displacement for a box barge


Deniz adds the Froude number and pushes:

In [12]:
%%writefile -a hydrostatics.py


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5

Appending to hydrostatics.py


In [13]:
!git add hydrostatics.py
!git commit -m "Add the Froude number"
!git push

[main 26236e2] Add the Froude number
 1 file changed, 5 insertions(+)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 20 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 398 bytes | 398.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
To /home/emircem/git-course/05-remotes/barge.git
   2dbd296..26236e2  main -> main


## Fetch: what's new?

Back in our repository, git does not know about Deniz's commit yet — it
never talks to the server unless you ask.

In [14]:
%cd ../barge
!git status

/home/emircem/git-course/05-remotes/barge
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


`git fetch` downloads new commits from the server, but does not change
your files:

In [15]:
!git fetch

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 378 bytes | 378.00 KiB/s, done.
From ../barge
   2dbd296..26236e2  main       -> origin/main


In [16]:
!git status

On branch main
Your branch is behind 'origin/main' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)

nothing to commit, working tree clean


In [17]:
!git log --oneline --all

26236e2 (origin/main, origin/HEAD) Add the Froude number
2dbd296 (HEAD -> main) Add volume and displacement for a box barge


`origin/main` is where the server's `main` is. Ours is one commit behind.

## Pull: fetch and merge

`git pull` fetches and then merges in one step:

In [18]:
!git pull

Updating 2dbd296..26236e2
Fast-forward
 hydrostatics.py | 5 +++++
 1 file changed, 5 insertions(+)


In [19]:
print(Path("hydrostatics.py").read_text())

"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5



## When both of you changed something

We add a function...

In [20]:
%%writefile -a hydrostatics.py


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 100000.0

Appending to hydrostatics.py


In [21]:
!git add hydrostatics.py
!git commit -m "Add tonnes per centimetre immersion"

[main 8358191] Add tonnes per centimetre immersion
 1 file changed, 5 insertions(+)


...and at the same time Deniz changes the first line and pushes:

In [22]:
%cd ../deniz

/home/emircem/git-course/05-remotes/deniz


In [23]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge, in SI units (m, kg, N)."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5

Overwriting hydrostatics.py


In [24]:
!git add hydrostatics.py
!git commit -m "Say which units the module uses"
!git push

[main 2c3595e] Say which units the module uses
 1 file changed, 1 insertion(+), 1 deletion(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 20 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 314 bytes | 314.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
To /home/emircem/git-course/05-remotes/barge.git
   26236e2..2c3595e  main -> main


Now we try to push:

In [25]:
%cd ../barge

/home/emircem/git-course/05-remotes/barge


In [26]:
# This one fails on purpose: the server has a commit we don't have.
!git push

To ../barge.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to '../barge.git'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


**Rejected.** The server will not throw away Deniz's commit to take ours.
We must get Deniz's work first. But a plain `git pull` refuses too, because
both sides have new commits and git wants us to choose how to join them:

In [27]:
# This one fails on purpose: git asks us to choose.
!git pull

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 294 bytes | 294.00 KiB/s, done.
From ../barge
   26236e2..2c3595e  main       -> origin/main
hint: You have divergent branches and need to specify how to reconcile them.
hint: You can do so by running one of the following commands sometime before
hint: your next pull:
hint:
hint:   git config pull.rebase false  # merge
hint:   git config pull.rebase true   # rebase
hint:   git config pull.ff only       # fast-forward only
hint:
hint: You can replace "git config" with "git config --global" to set a default
hint: preference for all repositories. You can also pass --rebase, --no-rebase,
hint: or --ff-only on the command line to override the configured default per
hint: invocation.
fatal: Need to specify how to reconcile divergent branches.


The two choices are chapters 3 and 4:

- `git pull --no-rebase` — join them with a merge commit.
- `git pull --rebase` — put our commit on top of Deniz's.

Our commit was never shared, so rebasing it is safe:

In [28]:
!git pull --rebase

Successfully rebased and updated refs/heads/main.


In [29]:
!git push

Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 20 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 419 bytes | 419.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
To ../barge.git
   2c3595e..25d6c01  main -> main


In [30]:
!git log --oneline --graph --all

* 25d6c01 (HEAD -> main, origin/main, origin/HEAD) Add tonnes per centimetre immersion
* 2c3595e Say which units the module uses
* 26236e2 Add the Froude number
* 2dbd296 Add volume and displacement for a box barge


## Pushing a branch

Branches stay on your computer until you push them:

In [31]:
!git switch -c wetted-surface

Switched to a new branch 'wetted-surface'


In [32]:
%%writefile -a hydrostatics.py


def wetted_surface(length, beam, draft):
    """Rough wetted surface area, in m^2."""
    return length * (beam + 2 * draft)

Appending to hydrostatics.py


In [33]:
!git add hydrostatics.py
!git commit -m "Add a rough wetted surface estimate"
!git push -u origin wetted-surface

[wetted-surface 55ada65] Add a rough wetted surface estimate
 1 file changed, 5 insertions(+)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 20 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 369 bytes | 369.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
To ../barge.git
 * [new branch]      wetted-surface -> wetted-surface
branch 'wetted-surface' set up to track 'origin/wetted-surface'.


In [34]:
!git branch -a

  main
* wetted-surface
  remotes/origin/HEAD -> origin/main
  remotes/origin/main
  remotes/origin/wetted-surface


On GitHub, this is when you would open a pull request (chapter 6).

## Summary

| I want to… | Command |
| - | - |
| copy a repository | `git clone <url>` |
| name a server | `git remote add origin <url>` |
| send my commits | `git push` |
| get new commits, without touching my files | `git fetch` |
| get new commits and merge them | `git pull` |
| get new commits, my work on top | `git pull --rebase` |
| push a new branch | `git push -u origin <branch>` |

Never use `git push --force` on a shared branch: it deletes other people's
commits from the server.

## Exercises

1. In `deniz`, run `git log --oneline --all` and compare with `barge`.
2. Make both repositories change the *same* line, then `git pull --rebase`
   and resolve the conflict.

Next: [6. Forks](06-forks.ipynb)